In [ ]:
# OFT Motion Projector + Tokenizer Adapter
# This file is a runnable script / notebook-style collection of functions and
# example cells to:
#  - provide a BaseMotionTokenizer API
#  - implement Classic (clustering) and VQ-VAE tokenizers (minimal placeholders)
#  - implement a MotionProjector PyTorch module compatible with OFT-style pipelines
#  - show a minimal fine-tuning loop with LoRA (using peft) to adapt the projector
#
# NOTE:
#  - This is a blueprint. Replace placeholder implementations (e.g., segmentation,
#    VQ-VAE encoder) with your real data / models.
#  - Requires: torch, sklearn, peft, transformers (for LLM wrapper), einops (optional)
#  - To run on CPU for small toy tests; use GPU for real runs.

# ======= Imports =======
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

# Optional imports for LoRA
try:
    from peft import get_peft_model, LoraConfig, TaskType
    PEFT_AVAILABLE = True
except Exception:
    PEFT_AVAILABLE = False



# ======= VQ-VAE Tokenizer (minimal placeholder) =======
class DummyVQVaeEncoder(nn.Module):
    def __init__(self, input_dim=3, latent_dim=64):
        super().__init__()
        # simple 1D conv encoder placeholder
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, latent_dim)
        )
    def forward(self, x):
        # x: (B, L, input_dim) -> reorder
        x = x.permute(0,2,1)
        return self.net(x)  # (B, latent_dim)

class VQVaeMotionTokenizer(BaseMotionTokenizer):
    def __init__(self, encoder, codebook_vectors):
        self.encoder = encoder
        self._codebook = np.asarray(codebook_vectors).astype(np.float32)

    def encode(self, trajectory):
        # segment into fixed length segments for encoder
        seg_len = 16
        ids = []
        for s in range(0, max(1, trajectory.shape[0] - seg_len + 1), seg_len):
            seg = trajectory[s:s+seg_len]
            with torch.no_grad():
                t = torch.from_numpy(seg.astype(np.float32)).unsqueeze(0)  # (1,L,3)
                z = self.encoder(t).cpu().numpy()[0]  # (latent_dim,)
            # nearest neighbor to codebook
            d = np.linalg.norm(self._codebook - z[None,:], axis=1)
            k = int(np.argmin(d))
            ids.append(k)
        return ids

    @property
    def codebook(self):
        return self._codebook

# ======= MotionProjector (OFT-compatible) =======
class MotionProjector(nn.Module):
    """
    Maps codebook vectors (K, D) to VLM embedding space (d_vlm).
    Usage:
      projector = MotionProjector(codebook=np_array_KD, d_vlm=4096)
      embeddings = projector(ids_tensor)  # (T, d_vlm)

    The codebook is stored inside the module for convenience but can be updated.
    """
    def __init__(self, codebook: np.ndarray, d_vlm: int = 4096, hidden_mult: int = 4,
                 codebook_trainable: bool = False):
        super().__init__()
        K, D = codebook.shape
        self.register_buffer('codebook', torch.tensor(codebook))
        if codebook_trainable:
            # make it trainable param
            self.codebook = nn.Parameter(self.codebook)
        hidden = max(D * hidden_mult, 128)
        self.proj = nn.Sequential(
            nn.Linear(D, hidden),
            nn.GELU(),
            nn.Linear(hidden, d_vlm),
            nn.LayerNorm(d_vlm)
        )

    def forward(self, ids: torch.LongTensor):
        # ids: (T,) or (B, T)
        single = False
        if ids.dim() == 1:
            single = True
            ids = ids.unsqueeze(0)
        emb = F.embedding(ids, self.codebook)  # (B, T, D)
        B, T, D = emb.shape
        emb = emb.view(B*T, D)
        out = self.proj(emb)
        out = out.view(B, T, -1)
        if single:
            out = out.squeeze(0)
        return out

    def set_codebook(self, new_codebook: np.ndarray):
        # replace buffer
        self.register_buffer('codebook', torch.tensor(new_codebook.astype(np.float32)))

# ======= LoRA integration helper (for single linear layers) =======
# This is a very small convenience wrapper if peft is unavailable
class TinyLoRA(nn.Module):
    def __init__(self, module: nn.Module, r=8, alpha=16):
        super().__init__()
        # only support linear module
        assert isinstance(module, nn.Linear)
        self.module = module
        in_dim, out_dim = module.in_features, module.out_features
        self.r = r
        self.alpha = alpha
        if r > 0:
            self.A = nn.Parameter(torch.zeros((r, in_dim)))
            self.B = nn.Parameter(torch.zeros((out_dim, r)))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            nn.init.zeros_(self.B)
            self.scale = alpha / r
        else:
            self.register_parameter('A', None)
            self.register_parameter('B', None)

    def forward(self, x):
        out = self.module(x)
        if self.r > 0:
            delta = (x @ self.A.T) @ self.B.T  # (B, out_dim)
            out = out + self.scale * delta
        return out



In [ ]:
# ======= Base Tokenizer API =======
class BaseMotionTokenizer:
    def encode(self, trajectory):
        """Return list[int] token ids for given trajectory (np.ndarray T x Dpoints)
        """
        raise NotImplementedError

    def decode(self, ids):
        """Optional: reconstruct sequence from ids
        """
        raise NotImplementedError

    @property
    def codebook(self):
        """Return numpy array (K, D) of codebook vectors
        """
        raise NotImplementedError

    @property
    def vocab_size(self):
        return int(self.codebook.shape[0])

# ======= Classic (DP + clustering) tokenizer =======
# Placeholder DP segmentation function - replace with your actual implementation

def douglas_peucker_segmentation(trajectory, epsilon=0.01):
    # Very naive segmentation: split into fixed-length chunks as placeholder
    T = trajectory.shape[0]
    seg_len = 16
    segments = []
    for s in range(0, max(1, T - seg_len + 1), seg_len):
        segments.append(trajectory[s:s+seg_len])
    return segments


def compute_handcrafted_features(segment):
    # segment: (L, 3) e.g. x,y,z
    # returns simple features: lin, L, d, ux, uy, uz, cx, cy, cz
    pts = np.asarray(segment)
    diffs = np.diff(pts, axis=0)
    step_lengths = np.linalg.norm(diffs, axis=1)
    L = float(np.sum(step_lengths))
    start = pts[0]
    end = pts[-1]
    d = float(np.linalg.norm(end - start))
    lin = float(d / (L + 1e-12))
    u = np.zeros(3)
    if d > 1e-9:
        u = (end - start) / (d + 1e-12)
    centroid = pts.mean(axis=0)
    feat = np.concatenate(([lin, L, d], u, centroid))
    return feat


class ClassicMotionTokenizer(BaseMotionTokenizer):
    def __init__(self, n_clusters=64, seg_len=16, dp_epsilon=0.01):
        self.n_clusters = n_clusters
        self.seg_len = seg_len
        self.dp_epsilon = dp_epsilon
        self.kmeans = None
        self._codebook = None

    def fit(self, list_of_trajectories):
        feats = []
        for traj in list_of_trajectories:
            segs = douglas_peucker_segmentation(traj, epsilon=self.dp_epsilon)
            for seg in segs:
                feats.append(compute_handcrafted_features(seg))
        F = np.vstack(feats)
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=0)
        self.kmeans.fit(F)
        self._codebook = self.kmeans.cluster_centers_.astype(np.float32)
        return self

    def encode(self, trajectory):
        segs = douglas_peucker_segmentation(trajectory, epsilon=self.dp_epsilon)
        ids = []
        for seg in segs:
            f = compute_handcrafted_features(seg).reshape(1, -1)
            ids.append(int(self.kmeans.predict(f)[0]))
        return ids

    def decode(self, ids):
        # decode as centroids (not a trajectory, just proto-feature)
        return self._codebook[ids]

    @property
    def codebook(self):
        return self._codebook



In [ ]:
import tensorflow_datasets as tfds
PATH = "./tf_datasets/columbia_cairlab_pusht_real/0.1.0"

# DATASET:
# Load builder from local directory
builder = tfds.builder_from_directory(builder_dir=PATH)

# Full train split; you can slice if you want (e.g. "train[:10]")
ds = builder.as_dataset(split="train")

episodes_displacements = {}  # ep_idx -> dict with 'instruction' and 'displacements'

for ep_idx, episode in enumerate(ds):
    # episode["steps"] is a tf.data.Dataset of step dicts
    step_ds = episode["steps"]

    disps = []

    # Convert steps to numpy for easy use
    for step in tfds.as_numpy(step_ds):
      # world_vector is shape (3,) float32: [dx, dy, dz]
        dv = step["action"]["world_vector"]  # numpy array of shape (3,)
        disps.append(dv)

    # Convert list of (3,) arrays to (T, 3) array
    disps = np.stack(disps, axis=0)  # shape (T, 3)

    episodes_displacements[ep_idx] = disps

In [ ]:
# TOKENIZER:
# Toy dataset: random trajectories + text
N = 200
trajs = [np.cumsum(np.random.randn(160,3)*0.05, axis=0) for _ in range(N)]
texts = ["pick up object" if i%2==0 else "move forward" for i in range(N)]

# Fit classic tokenizer
classic_tok = ClassicMotionTokenizer(n_clusters=32, seg_len=16)
classic_tok.fit(trajs)
ids0 = classic_tok.encode(trajs[0])
print('example ids classic:', ids0[:10])




In [ ]:
# Build projector
codebook = classic_tok.codebook  # (K, D)
projector = MotionProjector(codebook, d_vlm=512)

# Toy LLM embedding stub (simulate frozen VLM)
class DummyVLM(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        self.enc = nn.Linear(d_model, d_model)
    def forward(self, x):
        # x: (B, T, d_model)
        return x.mean(dim=1)
vlm = DummyVLM(d_model=512)

# If peft is available, attach LoRA to a dummy linear inside vlm (illustration):
if PEFT_AVAILABLE:
    lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=['enc'], task_type=TaskType.SEQ_2_SEQ_LM)
    vlm = get_peft_model(vlm, lora_config)

# Trainable params: projector.proj + optional LoRA inside vlm
opt = torch.optim.AdamW(list(projector.proj.parameters()) + ([] if not PEFT_AVAILABLE else list(vlm.parameters())), lr=1e-4)

# Minimal training: map motion embeddings to a toy target (e.g., text embedding simulation)
for epoch in range(3):
    total = 0.0
    for i in range(len(trajs)):
        ids = classic_tok.encode(trajs[i])
        ids_t = torch.tensor(ids, dtype=torch.long)
        emb = projector(ids_t)  # (T, d_vlm)
        if emb.dim() == 2:
            emb = emb.unsqueeze(0)
        out = vlm(emb)  # (B, d_vlm)
        # toy target
        target = torch.randn_like(out)
        loss = F.mse_loss(out, target)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print('epoch', epoch, 'loss', total/len(trajs))

print('Done toy fine-tune')

In [ ]:
# prismatic/models/projectors.py

class MotionProjector(nn.Module):
    def __init__(self, input_dim, llm_dim):
        super().__init__()
        # dim input_dim: dimension of codebook vectors
        self.proj = nn.Sequential(
            nn.Linear(input_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )

    def forward(self, x):
        # x shape: (batch, motion_len, input_dim)
        return self.proj(x)


In [ ]:


# Modifier OPENVLA CONFIG POUR AJOUTER MON MLP
# Modifier le fichier configuration_prismatic.py pour ajouter les paramètres du MLP dans la classe OpenVLAConfig
# Puis modifier la classe OpenVLAForActionPrediction dans le fichier model_prismatic.py pour intégrer le MLP dans le forward pass
# Enfin, modifier l'appel au modèle ci-dessous pour utiliser le nouveau modèle modifié





In [ ]:
import os
import time
from torch.nn.parallel import DistributedDataParallel as DDP
from typing import Dict, Optional, Tuple, Type
from pathlib import Path
from transformers.modeling_outputs import CausalLMOutputWithPast
from prismatic.training.train_utils import (
    compute_actions_l1_loss,
    compute_token_accuracy,
    get_current_action_mask,
    get_next_actions_mask,
)
from prismatic.vla.constants import (
    ACTION_DIM,
    ACTION_PROPRIO_NORMALIZATION_TYPE,
    NUM_ACTIONS_CHUNK,
    PROPRIO_DIM,
)
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics
import torch.distributed as dist
from transformers import AutoConfig, AutoImageProcessor, AutoModelForVision2Seq, AutoProcessor
from peft import LoraConfig, PeftModel, get_peft_model

# Configurations
vla_path: str = "openvla/openvla-7b"             # Path to OpenVLA model (on HuggingFace Hub or stored locally)
run_root_dir: Path = Path("runs")   
# Trim trailing forward slash ('/') in VLA path if it exists
vla_path = vla_path.rstrip("/")
print(f"Fine-tuning OpenVLA Model `{vla_path}` ")
num_images_in_input= 1 #number of images in input
use_lora=True
use_film=False
lora_rank: int = 32                              # Rank of LoRA weight matrix
lora_dropout: float = 0.0     
resume=False 
resume_step: Optional[int] = None                # (When `resume==True`) Step number that we are resuming from
use_proprio=False
use_l1_regression=False
use_diffusion=False
# continuous action head with diffusion modeling objective (DDIM)
num_diffusion_steps_train: int = 50              # (When `diffusion==True`) 
learning_rate: float = 5e-4                      # Learning rate
num_steps_before_decay: int = 100_000            # Number of steps before LR decays by 10x
  # Dataset
data_root_dir: Path = Path("tf_datasets") 
dataset_name: str = "columbia_cairlab_pusht_real/0.1.0"    # Name of fine-tuning dataset (e.g., `aloha_scoop_x_into_bowl`)
shuffle_buffer_size: int = 100_000               # Dataloader shuffle buffer size (can reduce if OOM errors occur)
image_aug: bool = True                           # If True, trains with image augmentations (HIGHLY RECOMMENDED)
use_val_set: bool = False                        # If True, uses validation set and log validation metrics
batch_size: int = 8    
grad_accumulation_steps: int = 1                 # Number of gradient accumulation steps
max_steps: int = 200_000                         # Max number of training steps
diffusion_sample_freq: int = 50                  # (When `use_diffusion==True`) Frequency for sampling in steps
lr_warmup_steps: int = 0                         # Number of steps to warm up learning rate (from 10% to 100%)
save_freq: int = 10_000                          # Checkpoint saving frequency in steps
val_freq: int = 10_000                           # (When `use_val_set==True`) Validation set logging frequency in steps
val_time_limit: int = 180                        # (When `use_val_set==True`) Time limit for computing validation metrics
save_latest_checkpoint_only: bool = False        # If True, saves only 1 checkpoint, overwriting latest checkpoint
merge_lora_during_training: bool = True          # If True, merges LoRA weights and saves result during training
import wandb

def log_metrics_to_wandb(metrics, prefix, step, wandb_entity) -> None:
    """
    Log metrics to Weights & Biases.

    Args:
        metrics (dict): Dictionary of metrics to log
        prefix (str): Prefix for metric names
        step (int): Training step
        wandb_entity (str): W&B entity instance

    Returns:
        None.
    """
    log_dict = {}
    for name, value in metrics.items():
        # Map loss_value to Loss for better readability in W&B
        if name == "loss_value":
            log_dict[f"{prefix}/Loss"] = value
        # Keep other metrics as is
        else:
            log_dict[f"{prefix}/{name.replace('_', ' ').title()}"] = value
    wandb_entity.log(log_dict, step=step)



def compute_smoothened_metrics(metrics_deques) -> dict:
    """
    Compute smoothened metrics from recent deques.

    Args:
        metrics_deques (dict): Dictionary of deques containing recent metrics.

    Returns:
        dict: Dictionary of smoothened metrics.
    """
    smoothened_metrics = {}
    for name, deque in metrics_deques.items():
        if deque and len(deque) > 0:
            smoothened_metrics[name] = sum(deque) / len(deque)
    return smoothened_metrics

def load_checkpoint(module_name: str, path: str, step: int, device: str = "cpu") -> dict:
    """
    Loads a checkpoint for a given module.

    Args:
        module_name (str): Name of model component to load checkpoint for.
        path (str): Path to checkpoint directory.
        step (int): Gradient step number of saved checkpoint.
        device (str): String specifying how to remap storage locations (default = "cpu").

    Returns:
        dict: PyTorch model state dictionary.
    """
    checkpoint_path = os.path.join(path, f"{module_name}--{step}_checkpoint.pt")
    print(f"Loading checkpoint: {checkpoint_path}")
    state_dict = torch.load(checkpoint_path, weights_only=True, map_location=device)
    return remove_ddp_in_checkpoint(state_dict)

def remove_ddp_in_checkpoint(state_dict) -> dict:
    """
    Removes the 'module.' prefix from parameter names in a PyTorch model state dictionary that was saved using
    DistributedDataParallel (DDP).

    When a model is trained using PyTorch's DistributedDataParallel, the saved state dictionary contains parameters
    prefixed with 'module.'. This function removes these prefixes to make the state dictionary compatible when
    loading into models that are not yet wrapped in DDP.

    Args:
        state_dict (dict): PyTorch model state dictionary.

    Returns:
        dict: A new state dictionary with the same contents but with 'module.' prefixes removed from parameter names.
              Parameters without the 'module.' prefix remain unchanged.
    """
    new_state_dict = {}
    for k, v in state_dict.items():
        if k[:7] == "module.":
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    return new_state_dict

def wrap_ddp(module: nn.Module, device_id: int, find_unused: bool = False) -> DDP:
    """
    Wrap a module with DistributedDataParallel.

    Args:
        module (nn.Module): PyTorch module.
        device_id (str): Device ID.
        find_unused (bool): Whether to detect parameters without gradients in distributed training.

    Returns:
        DistributedDataParallel: PyTorch module wrapped with DDP.
    """
    return DDP(module, device_ids=[device_id], find_unused_parameters=find_unused, gradient_as_bucket_view=True)

def init_module(
    module_class: Type[nn.Module],
    module_name: str,
    device_id: int,
    module_args: dict,
    to_bf16: bool = False,
    find_unused_params: bool = False,
) -> DDP:
    """
    Initializes a module, optionally loads checkpoint, moves to device, and wraps with DDP.

    Args:
        module_class (Type[nn.Module]): Class of PyTorch module to initialize.
        module_name (str): Name of model component to load checkpoint for.
        cfg (FinetuneConfig): Training configuration.
        device_id (str): Device ID.
        module_args (dict): Args for initializing the module.
        to_bf16 (bool): Whether to convert to torch.bfloat16 data type.
        find_unused_params (bool): Whether to detect parameters without gradients in distributed training.

    Returns:
        DistributedDataParallel: PyTorch module wrapped with DDP.
    """
    module = module_class(**module_args)


    if resume:
        state_dict = load_checkpoint(module_name, vla_path, resume_step)
        module.load_state_dict(state_dict)

    if to_bf16:
        module = module.to(torch.bfloat16)
    module = module.to(device_id)

    return wrap_ddp(module, device_id, find_unused_params)


def run_diffusion_sampling(
    vla,
    action_head,
    noisy_action_projector,
    proprio_projector,
    batch,
    batch_size,
    num_patches,
    actions_shape,
    device_id,
    current_action_mask,
    next_actions_mask,
    use_proprio,
    use_film,
) -> torch.Tensor:
    """
    Run diffusion sampling (reverse diffusion) to generate actions.

    Args:
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        action_head (nn.Module): Action head module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        proprio_projector (nn.Module): Proprioceptive state projector module.
        batch (dict): Input batch.
        batch_size (int): Batch size.
        num_patches (int): Number of vision patches.
        actions_shape (tuple): Shape of ground-truth actions.
        device_id (str): Device ID.
        current_action_mask (torch.Tensor): Mask for current action.
        next_actions_mask (torch.Tensor): Mask for next actions.
        use_proprio (bool): Whether to use proprioceptive state as input.
        use_film (bool): Whether to use FiLM for better language following.

    Returns:
        torch.Tensor: Predicted actions.
    """
    # Sample random noisy action, used as the starting point for reverse diffusion
    noise = torch.randn(
        size=(batch_size, NUM_ACTIONS_CHUNK, ACTION_DIM),
        device=device_id,
        dtype=torch.bfloat16,
    )  # (B, chunk_len, action_dim)

    # Set diffusion timestep values
    action_head.module.noise_scheduler.set_timesteps(action_head.module.num_diffusion_steps_train)

    # Reverse diffusion: Iteratively denoise to generate action, conditioned on observation
    curr_noisy_actions = noise
    for t in action_head.module.noise_scheduler.timesteps:
        # Get diffusion model's noise prediction (conditioned on VLA latent embedding, current noisy action embedding,
        # and diffusion timestep embedding)
        timesteps = torch.Tensor([t]).repeat(batch_size).to(device_id)
        diffusion_timestep_embeddings = (
            action_head.module.time_encoder(timesteps).to(curr_noisy_actions.dtype).to(curr_noisy_actions.device)
        )  # (B, llm_dim)
        diffusion_timestep_embeddings = diffusion_timestep_embeddings.unsqueeze(1)  # (B, 1, llm_dim)

        with torch.autocast("cuda", dtype=torch.bfloat16):
            output = vla(
                input_ids=batch["input_ids"].to(device_id),
                attention_mask=batch["attention_mask"].to(device_id),
                pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
                labels=batch["labels"],
                output_hidden_states=True,
                proprio=batch["proprio"] if use_proprio else None,
                proprio_projector=proprio_projector if use_proprio else None,
                noisy_actions=curr_noisy_actions,
                noisy_action_projector=noisy_action_projector,
                diffusion_timestep_embeddings=diffusion_timestep_embeddings,
                use_film=use_film,
            )
            # Get last layer hidden states
            last_hidden_states = output.hidden_states[-1]  # (B, seq_len, D)
            # Get hidden states for text portion of prompt+response (after the vision patches)
            text_hidden_states = last_hidden_states[:, num_patches:-1]
            # Get hidden states for action portion of response
            actions_hidden_states = text_hidden_states[current_action_mask | next_actions_mask].reshape(
                batch_size, NUM_ACTIONS_CHUNK * ACTION_DIM, -1
            )  # (B, act_chunk_len, D)
            actions_hidden_states = actions_hidden_states.to(torch.bfloat16)
            # Predict noise
            noise_pred = action_head.module.predict_noise(actions_hidden_states)

        # Compute the action at the previous diffusion timestep: x_t -> x_{t-1}
        curr_noisy_actions = action_head.module.noise_scheduler.step(noise_pred, t, curr_noisy_actions).prev_sample

    return curr_noisy_actions.reshape(actions_shape)

def run_forward_pass(
    vla,
    action_head,
    noisy_action_projector,
    proprio_projector,
    batch,
    action_tokenizer,
    device_id,
    use_l1_regression,
    use_diffusion,
    use_proprio,
    use_film,
    num_patches,
    compute_diffusion_l1=False,
    num_diffusion_steps_train=None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Compute model forward pass and metrics for both training and validation.

    Args:
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        action_head (nn.Module): Action head module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        proprio_projector (nn.Module): Proprioceptive state projector module.
        batch (dict): Input batch.
        action_tokenizer (ActionTokenizer): Action tokenizer.
        device_id (str): Device ID.
        use_l1_regression (bool): Whether to use L1 regression.
        use_diffusion (bool): Whether to use diffusion.
        use_proprio (bool): Whether to use proprioceptive state as input.
        use_film (bool): Whether to use FiLM for better language following.
        num_patches (int): Number of vision patches.
        compute_diffusion_l1 (bool): Whether to sample actions and compute L1 loss for diffusion (do this once every
                                    diffusion_sample_freq steps during training; do it every batch for validation)
        num_diffusion_steps_train (int): Number of diffusion steps for training (only used for diffusion).

    Returns:
        tuple: (loss, metrics_dict)
            loss: The loss tensor with gradient for backpropagation.
            metrics_dict: Dictionary of computed metrics (detached values for logging).
    """
    metrics = {}

    # Get ground-truth action labels
    ground_truth_actions = batch["actions"].to(device_id).to(torch.bfloat16)

    # [Only for diffusion] Sample noisy actions used as input for noise predictor network
    if use_diffusion:
        noisy_dict = action_head.module.sample_noisy_actions(ground_truth_actions)
        noise, noisy_actions, diffusion_timestep_embeddings = (
            noisy_dict["noise"],
            noisy_dict["noisy_actions"],
            noisy_dict["diffusion_timestep_embeddings"],
        )
    else:
        noise, noisy_actions, diffusion_timestep_embeddings = None, None, None

    # VLA forward pass
    with torch.autocast("cuda", dtype=torch.bfloat16):
        output: CausalLMOutputWithPast = vla(
            input_ids=batch["input_ids"].to(device_id),
            attention_mask=batch["attention_mask"].to(device_id),
            pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
            labels=batch["labels"],
            output_hidden_states=True,
            proprio=batch["proprio"] if use_proprio else None,
            proprio_projector=proprio_projector if use_proprio else None,
            noisy_actions=noisy_actions if use_diffusion else None,
            noisy_action_projector=noisy_action_projector if use_diffusion else None,
            diffusion_timestep_embeddings=diffusion_timestep_embeddings if use_diffusion else None,
            use_film=use_film,
        )

    # Get action masks needed for logging
    ground_truth_token_ids = batch["labels"][:, 1:].to(device_id)
    current_action_mask = get_current_action_mask(ground_truth_token_ids)
    next_actions_mask = get_next_actions_mask(ground_truth_token_ids)

    # Compute metrics for discrete action representation (next-token prediction)
    if not (use_l1_regression or use_diffusion):
        loss = output.loss
        predicted_token_ids = output.logits[:, num_patches:-1].argmax(dim=2)
        curr_action_accuracy = compute_token_accuracy(
            predicted_token_ids, ground_truth_token_ids, mask=current_action_mask
        )
        curr_action_l1_loss = compute_actions_l1_loss(
            action_tokenizer, predicted_token_ids, ground_truth_token_ids, mask=current_action_mask
        )
        next_actions_accuracy = compute_token_accuracy(
            predicted_token_ids, ground_truth_token_ids, mask=next_actions_mask
        )
        next_actions_l1_loss = compute_actions_l1_loss(
            action_tokenizer, predicted_token_ids, ground_truth_token_ids, mask=next_actions_mask
        )
        metrics.update(
            {
                "loss_value": loss.item(),  # Detached value for logging
                "curr_action_accuracy": curr_action_accuracy.item(),
                "curr_action_l1_loss": curr_action_l1_loss.item(),
                "next_actions_accuracy": next_actions_accuracy.item(),
                "next_actions_l1_loss": next_actions_l1_loss.item(),
            }
        )
    # Compute metrics for continuous action representations (L1 regression | diffusion)
    else:
        # Get last layer hidden states
        last_hidden_states = output.hidden_states[-1]  # (B, seq_len, D)
        # Get hidden states for text portion of prompt+response (after the vision patches)
        text_hidden_states = last_hidden_states[:, num_patches:-1]
        # Get hidden states for action portion of response
        batch_size = batch["input_ids"].shape[0]
        actions_hidden_states = (
            text_hidden_states[current_action_mask | next_actions_mask]
            .reshape(batch_size, NUM_ACTIONS_CHUNK * ACTION_DIM, -1)
            .to(torch.bfloat16)
        )  # (B, act_chunk_len, D)

        if use_l1_regression:
            # Predict action
            predicted_actions = action_head.module.predict_action(actions_hidden_states)
            # Get full L1 loss
            loss = torch.nn.L1Loss()(ground_truth_actions, predicted_actions)

        if use_diffusion:
            # Predict noise
            noise_pred = action_head.module.predict_noise(actions_hidden_states)
            # Get diffusion noise prediction MSE loss
            noise_pred = noise_pred.reshape(noise.shape)
            loss = nn.functional.mse_loss(noise_pred, noise, reduction="mean")

            # Only sample actions and compute L1 losses if specified
            if compute_diffusion_l1:
                with torch.no_grad():
                    predicted_actions = run_diffusion_sampling(
                        vla=vla,
                        action_head=action_head,
                        noisy_action_projector=noisy_action_projector,
                        proprio_projector=proprio_projector,
                        batch=batch,
                        batch_size=batch_size,
                        num_patches=num_patches,
                        actions_shape=ground_truth_actions.shape,
                        device_id=device_id,
                        current_action_mask=current_action_mask,
                        next_actions_mask=next_actions_mask,
                        use_proprio=use_proprio,
                        use_film=use_film,
                    )

        metrics.update(
            {
                "loss_value": loss.item(),  # Detached value for logging
            }
        )

        # Get detailed L1 losses for logging
        should_log_l1_loss = not use_diffusion or (use_diffusion and compute_diffusion_l1)
        if should_log_l1_loss:
            ground_truth_curr_action = ground_truth_actions[:, 0]
            predicted_curr_action = predicted_actions[:, 0]
            ground_truth_next_actions = ground_truth_actions[:, 1:]
            predicted_next_actions = predicted_actions[:, 1:]
            curr_action_l1_loss = torch.nn.L1Loss()(ground_truth_curr_action, predicted_curr_action)
            next_actions_l1_loss = torch.nn.L1Loss()(ground_truth_next_actions, predicted_next_actions)
            metrics.update(
                {
                    "curr_action_l1_loss": curr_action_l1_loss.item(),
                    "next_actions_l1_loss": next_actions_l1_loss.item(),
                }
            )

    # Return both the loss tensor (with gradients) and the metrics dictionary (with detached values)
    return loss, metrics

def save_training_checkpoint(
    run_dir,
    log_step,
    vla,
    processor,
    proprio_projector,
    noisy_action_projector,
    action_head,
    train_dataset,
    distributed_state,
) -> None:
    """
    Save all training checkpoints including model components, LoRA adapter, and dataset statistics.

    Args:
        cfg (FinetuneConfig): Training configuration.
        run_dir (Path): Experiment run directory path.
        log_step (int): Current logging step.
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        processor (PrismaticProcessor): OpenVLA inputs processor.
        proprio_projector (nn.Module): Proprioceptive state projector module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        action_head (nn.Module): Action head module.
        train_dataset (RLDSDataset): Training dataset.
        distributed_state (PartialState): Distributed training state.

    Returns:
        None.
    """
    # Determine checkpoint paths and naming
    if save_latest_checkpoint_only:
        checkpoint_dir = run_dir
        checkpoint_name_suffix = "latest_checkpoint.pt"
    else:
        checkpoint_dir = Path(str(run_dir) + f"--{log_step}_chkpt")
        checkpoint_name_suffix = f"{log_step}_checkpoint.pt"

    adapter_dir = checkpoint_dir / "lora_adapter"

    # Create directories and save dataset statistics (main process only)
    if distributed_state.is_main_process:
        os.makedirs(checkpoint_dir, exist_ok=True)
        os.makedirs(adapter_dir, exist_ok=True)
        save_dataset_statistics(train_dataset.dataset_statistics, checkpoint_dir)
        print(f"Saving Model Checkpoint for Step {log_step}")

    # Wait for directories to be created
    dist.barrier()

    # Save model components (main process only)
    if distributed_state.is_main_process:
        # Save processor and LoRA adapter
        processor.save_pretrained(checkpoint_dir)
        vla.module.save_pretrained(adapter_dir)

        # Save other components
        if use_proprio and proprio_projector is not None:
            torch.save(proprio_projector.state_dict(), checkpoint_dir / f"proprio_projector--{checkpoint_name_suffix}")

        if use_diffusion and noisy_action_projector is not None:
            torch.save(
                noisy_action_projector.state_dict(), checkpoint_dir / f"noisy_action_projector--{checkpoint_name_suffix}"
            )

        if (use_l1_regression or use_diffusion) and action_head is not None:
            torch.save(action_head.state_dict(), checkpoint_dir / f"action_head--{checkpoint_name_suffix}")

        if use_film:
            # To be safe, just save the entire vision backbone (not just FiLM components)
            torch.save(
                vla.module.vision_backbone.state_dict(), checkpoint_dir / f"vision_backbone--{checkpoint_name_suffix}"
            )

    # Wait for model components to be saved
    dist.barrier()

    # Merge LoRA weights into base model and save resulting model checkpoint
    # Note: Can be very slow on some devices; if so, we recommend merging offline
    if use_lora and merge_lora_during_training:
        base_vla = AutoModelForVision2Seq.from_pretrained(
            vla_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
        )
        merged_vla = PeftModel.from_pretrained(base_vla, adapter_dir)
        merged_vla = merged_vla.merge_and_unload()

        if distributed_state.is_main_process:
            merged_vla.save_pretrained(checkpoint_dir)
            print(f"Saved merged model for Step {log_step} at: {checkpoint_dir}")

        # Wait for merged model to be saved
        dist.barrier()

def run_validation(
    vla,
    action_head,
    noisy_action_projector,
    proprio_projector,
    val_dataloader,
    action_tokenizer,
    device_id,
    num_patches,
    log_step,
    distributed_state,
    val_time_limit,
) -> None:
    """
    Compute validation set metrics for logging.

    Args:
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        action_head (nn.Module): Action head module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        proprio_projector (nn.Module): Proprioceptive state projector module.
        val_dataloader (DataLoader): Validation data loader.
        action_tokenizer (ActionTokenizer): Action tokenizer.
        device_id (str): Device ID.
        cfg (FinetuneConfig): Training configuration.
        num_patches (int): Number of vision patches.
        log_step (int): Current logging step.
        distributed_state (PartialState): Distributed training state.
        val_time_limit (int): Time limit for computing validation metrics.

    Returns:
        None.
    """
    val_start_time = time.time()
    vla.eval()
    val_batches_count = 0

    # List to store validation metrics
    all_val_metrics = []

    with torch.no_grad():
        for batch in val_dataloader:
            # Always compute L1 loss for validation, even for diffusion
            _, metrics = run_forward_pass(
                vla=vla,
                action_head=action_head,
                noisy_action_projector=noisy_action_projector,
                proprio_projector=proprio_projector,
                batch=batch,
                action_tokenizer=action_tokenizer,
                device_id=device_id,
                use_l1_regression=use_l1_regression,
                use_diffusion=use_diffusion,
                use_proprio=use_proprio,
                use_film=use_film,
                num_patches=num_patches,
                compute_diffusion_l1=True,
                num_diffusion_steps_train=num_diffusion_steps_train if use_diffusion else None,
            )
 
            # Add the loss value to the metrics
            metrics["loss"] = metrics["loss_value"]
            all_val_metrics.append(metrics)
            val_batches_count += 1

            # Cut testing on validation set short if it exceeds time limit
            if time.time() - val_start_time > val_time_limit:
                break

    # Compute average validation metrics
    avg_val_metrics = {}
    for metric_name in all_val_metrics[0].keys():
        values = [metrics[metric_name] for metrics in all_val_metrics if metric_name in metrics]
        if values:
            avg_val_metrics[metric_name] = sum(values) / len(values)

    # Add batch count to metrics
    avg_val_metrics["val_batches_count"] = val_batches_count

    # Log validation metrics to W&B
    if distributed_state.is_main_process:
        log_metrics_to_wandb(avg_val_metrics, "VLA Val", log_step, wandb)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# FINE-TUNNING


from huggingface_hub import HfApi, snapshot_download 

from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.processing_prismatic import PrismaticImageProcessor, PrismaticProcessor
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction

import wandb
from prismatic.models.film_vit_wrapper import FiLMedPrismaticVisionBackbone
from prismatic.models.projectors import (
    NoisyActionProjector,
    ProprioProjector,
)
from prismatic.vla.constants import (
    ACTION_DIM,
    ACTION_PROPRIO_NORMALIZATION_TYPE,
    NUM_ACTIONS_CHUNK,
    PROPRIO_DIM
)
from prismatic.models.action_heads import DiffusionActionHead, L1RegressionActionHead
from torch.optim import AdamW
from torch.optim.lr_scheduler import MultiStepLR
from prismatic.vla.action_tokenizer import ActionTokenizer
from prismatic.vla.datasets import RLDSBatchTransform, RLDSDataset
from prismatic.models.backbones.llm.prompting import PurePromptBuilder
from accelerate import PartialState
from experiments.robot.openvla_utils import (
    check_model_logic_mismatch,
    model_is_on_hf_hub,
    update_auto_map,
)
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics
from torch.utils.data import DataLoader
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from torch.utils.data import DataLoader
from collections import deque
import tqdm



# Logging
wandb_entity: str = "your-wandb-entity"          # Name of WandB entity
wandb_project: str = "your-wandb-project"        # Name of WandB project
wandb_log_freq: int = 10                         # WandB logging frequency in steps
run_id=1


# Create experiment run directory
run_dir = run_root_dir / run_id
os.makedirs(run_dir, exist_ok=True)




device_id=0 #I will use only one GPU for the momment



 # GPU setup
distributed_state = PartialState()
device_id = distributed_state.local_process_index
torch.cuda.set_device(device_id)
torch.cuda.empty_cache()

# Initialize wandb logging
if distributed_state.is_main_process:
    wandb.init(entity=wandb_entity, project=wandb_project, name=f"ft+{run_id}")


# Two options:
# (1) Base model is on Hugging Face Hub
#   - Then download it and record the path to the download directory
# (2) Base model is stored locally
#   - Then register model config in HF Auto Classes
# In both cases, we want to check whether any changes have been made to
# the `modeling_prismatic.py` file in this codebase; if so, we will copy
# the file to the downloaded or locally stored checkpoint directory so
# that the user's changes to the VLA class logic go into effect
if model_is_on_hf_hub(vla_path):
    # Download model directly from Hugging Face Hub
    vla_download_path = snapshot_download(repo_id=vla_path)
    # Overwrite VLA path
    vla_path = vla_download_path
else:
    # Register OpenVLA model to HF Auto Classes (not needed if the model is on HF Hub)
    AutoConfig.register("openvla", OpenVLAConfig)
    AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
    AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
    AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)


# Update config.json and sync model files
if distributed_state.is_main_process:
    update_auto_map(vla_path)
    check_model_logic_mismatch(vla_path)

# Wait for model files to be synced
dist.barrier()


# Load processor and VLA
processor = AutoProcessor.from_pretrained(vla_path, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    vla_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to(device_id)

# Set number of images in VLA input
vla.vision_backbone.set_num_images_in_input(num_images_in_input)

# LoRA setup
if use_lora:
    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=min(lora_rank, 16),
        lora_dropout=lora_dropout,
        target_modules="all-linear",
        init_lora_weights="gaussian",
    )
    vla = get_peft_model(vla, lora_config)
    vla.print_trainable_parameters()

# FiLM setup
if use_film:
    # Wrap vision backbone with FiLM wrapper
    # Important: For this, must specify `vla.model.vision_backbone` instead of just `vla.vision_backbone`, since the
    # latter would cause the new wrapped backbone to be saved as a new attribute of `vla` instead of overwriting the
    # original one (due to the LoRA wrapper)
    vla.model.vision_backbone = FiLMedPrismaticVisionBackbone(
        vision_backbone=vla.model.vision_backbone,
        llm_dim=vla.llm_dim,
    )
    if resume:
        state_dict = load_checkpoint("vision_backbone", vla_path,resume_step)
        vla.model.vision_backbone.load_state_dict(state_dict)
    vla.model.vision_backbone = vla.model.vision_backbone.to(device_id)

# Wrap VLA with DDP
vla = wrap_ddp(vla, device_id, find_unused=True)

# If applicable, instantiate proprio projector
if use_proprio:
    proprio_projector = init_module(
        ProprioProjector,
        "proprio_projector",
        device_id,
        {"llm_dim": vla.module.llm_dim, "proprio_dim": PROPRIO_DIM},
    )

# If applicable, instantiate continuous action head for L1 regression
if use_l1_regression:
    action_head = init_module(
        L1RegressionActionHead,
        "action_head",
        device_id,
        {"input_dim": vla.module.llm_dim, "hidden_dim": vla.module.llm_dim, "action_dim": ACTION_DIM},
        to_bf16=True,
    )

# If applicable, instantiate diffusion action head and noisy action projector
if use_diffusion:
    action_head = init_module(
        DiffusionActionHead,
        "action_head",
        device_id,
        {
            "input_dim": vla.module.llm_dim,
            "hidden_dim": vla.module.llm_dim,
            "action_dim": ACTION_DIM,
            "num_diffusion_steps_train": num_diffusion_steps_train,
        },
        to_bf16=True,
    )
    noisy_action_projector = init_module(
        NoisyActionProjector, "noisy_action_projector", device_id, {"llm_dim": vla.module.llm_dim}
    )

# Get number of vision patches
NUM_PATCHES = vla.module.vision_backbone.get_num_patches() * vla.module.vision_backbone.get_num_images_in_input()
# If we have proprio inputs, a single proprio embedding is appended to the end of the vision patch embeddings
if use_proprio:
    NUM_PATCHES += 1
# For diffusion, a single diffusion timestep embedding is appended to the end of the vision patch embeddings
if use_diffusion:
    NUM_PATCHES += 1

# Instantiate optimizer
trainable_params = [param for param in vla.parameters() if param.requires_grad]
if use_l1_regression or use_diffusion:
    trainable_params += [param for param in action_head.parameters() if param.requires_grad]
if use_diffusion:
    trainable_params += [param for param in noisy_action_projector.parameters() if param.requires_grad]
if use_proprio:
    trainable_params += [param for param in proprio_projector.parameters() if param.requires_grad]
print(f"# total trainable params: {sum(p.numel() for p in trainable_params)}")
optimizer = AdamW(trainable_params, lr=learning_rate)

# Record original learning rate
original_lr = optimizer.param_groups[0]["lr"]

# Create learning rate scheduler
scheduler = MultiStepLR(
    optimizer,
    milestones=[num_steps_before_decay],  # Number of steps after which LR will change
    gamma=0.1,  # Multiplicative factor of learning rate decay
)

# Create Action Tokenizer
action_tokenizer = ActionTokenizer(processor.tokenizer)

# Load Fine-tuning Dataset =>> note that we use an RLDS-formatted dataset following Open X-Embodiment by default.
#   =>> If you want to use a non-RLDS dataset (e.g., a standard PyTorch Dataset) see the following commented block.
#   =>> Note that our training code does not loop over epochs because the RLDS loader does this implicitly; if using
#       your own Dataset, make sure to add the appropriate logic to the training loop!
#
# ---
# from prismatic.vla.datasets import DummyDataset
#
# train_dataset = DummyDataset(
#     action_tokenizer,
#     processor.tokenizer,
#     image_transform=processor.image_processor.apply_transform,
#     prompt_builder_fn=PurePromptBuilder,
# )
# ---

# We assume that the model takes as input one third-person camera image and 1 or 2 optional wrist camera image(s)
use_wrist_image = num_images_in_input > 1

# Create training and optional validation datasets
batch_transform = RLDSBatchTransform(
    action_tokenizer,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder,
    use_wrist_image=use_wrist_image,
    use_proprio=use_proprio,
)
train_dataset = RLDSDataset(
    data_root_dir,
    dataset_name,
    batch_transform,
    resize_resolution=tuple(vla.module.config.image_sizes),
    shuffle_buffer_size=shuffle_buffer_size,
    image_aug=image_aug,
)
if use_val_set:
    val_dataset = RLDSDataset(
        data_root_dir,
        dataset_name,
        batch_transform,
        resize_resolution=tuple(vla.module.config.image_sizes),
        shuffle_buffer_size=shuffle_buffer_size // 10,
        image_aug=image_aug,
        train=False,
    )

# [Important] Save dataset statistics so that we can unnormalize actions during inference
if distributed_state.is_main_process:
    save_dataset_statistics(train_dataset.dataset_statistics, run_dir)

# Create collator and dataloader
collator = PaddedCollatorForActionPrediction(
    processor.tokenizer.model_max_length, processor.tokenizer.pad_token_id, padding_side="right"
)
dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=None,
    collate_fn=collator,
    num_workers=0,  # Important: Set to 0 if using RLDS, which uses its own parallelism
)
if use_val_set:
    val_batch_size = batch_size
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=val_batch_size,
        sampler=None,
        collate_fn=collator,
        num_workers=0,  # Important: Set to 0 if using RLDS, which uses its own parallelism
    )

# Deque to store recent train metrics (used for computing smoothened metrics for gradient accumulation)
recent_metrics = {
    "loss_value": deque(maxlen=grad_accumulation_steps),
    "curr_action_accuracy": deque(maxlen=grad_accumulation_steps),
    "curr_action_l1_loss": deque(maxlen=grad_accumulation_steps),
    "next_actions_accuracy": deque(maxlen=grad_accumulation_steps),
    "next_actions_l1_loss": deque(maxlen=grad_accumulation_steps),
}

# Start training
with tqdm.tqdm(total=max_steps, leave=False) as progress:
    vla.train()
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(dataloader):
        # Compute training metrics and loss
        compute_diffusion_l1 = use_diffusion and batch_idx % diffusion_sample_freq == 0
        loss, metrics = run_forward_pass(
            vla=vla,
            action_head=action_head,
            noisy_action_projector=noisy_action_projector if use_diffusion else None,
            proprio_projector=proprio_projector if use_proprio else None,
            batch=batch,
            action_tokenizer=action_tokenizer,
            device_id=device_id,
            use_l1_regression=use_l1_regression,
            use_diffusion=use_diffusion,
            use_proprio=use_proprio,
            use_film=use_film,
            num_patches=NUM_PATCHES,
            compute_diffusion_l1=compute_diffusion_l1,
            num_diffusion_steps_train=num_diffusion_steps_train if use_diffusion else None,
        )

        # Normalize loss to account for gradient accumulation
        normalized_loss = loss / grad_accumulation_steps

        # Backward pass
        normalized_loss.backward()

        # Store recent train metrics
        for metric_name, value in metrics.items():
            if metric_name in recent_metrics:
                recent_metrics[metric_name].append(value)

        # Compute gradient step index
        gradient_step_idx = batch_idx // grad_accumulation_steps

        # Compute smoothened train metrics
        smoothened_metrics = compute_smoothened_metrics(recent_metrics)

        # Push Metrics to W&B (every wandb_log_freq gradient steps)
        log_step = gradient_step_idx if not resume else resume_step + gradient_step_idx
        if distributed_state.is_main_process and log_step % wandb_log_freq == 0:
            log_metrics_to_wandb(smoothened_metrics, "VLA Train", log_step, wandb)

        # [If applicable] Linearly warm up learning rate from 10% to 100% of original
        if lr_warmup_steps > 0:
            lr_progress = min((gradient_step_idx + 1) / lr_warmup_steps, 1.0)  # Cap at 1.0
            current_lr = original_lr * (0.1 + 0.9 * lr_progress)
            for param_group in optimizer.param_groups:
                param_group["lr"] = current_lr

        if distributed_state.is_main_process and gradient_step_idx % wandb_log_freq == 0:
            # Log the learning rate
            # Make sure to do this AFTER any learning rate modifications (e.g., warmup/decay)
            wandb.log(
                {
                    "VLA Train/Learning Rate": scheduler.get_last_lr()[0],
                },
                step=log_step,
            )

        # Optimizer and LR scheduler step
        if (batch_idx + 1) % grad_accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            progress.update()

        # Save model checkpoint: either keep latest checkpoint only or all checkpoints
        if gradient_step_idx > 0 and log_step % save_freq == 0:
            save_training_checkpoint(
                run_dir=run_dir,
                log_step=log_step,
                vla=vla,
                processor=processor,
                proprio_projector=proprio_projector if use_proprio else None,
                noisy_action_projector=noisy_action_projector if use_diffusion else None,
                action_head=action_head if (use_l1_regression or use_diffusion) else None,
                train_dataset=train_dataset,
                distributed_state=distributed_state,
            )

        # Test model on validation set
        if use_val_set and log_step > 0 and log_step % val_freq == 0:
            run_validation(
                vla=vla,
                action_head=action_head,
                noisy_action_projector=noisy_action_projector if use_diffusion else None,
                proprio_projector=proprio_projector if use_proprio else None,
                val_dataloader=val_dataloader,
                action_tokenizer=action_tokenizer,
                device_id=device_id,
                num_patches=NUM_PATCHES,
                log_step=log_step,
                distributed_state=distributed_state,
                val_time_limit=val_time_limit,
            )
            # Set model back to training mode after validation
            vla.train()

        # Stop training when max_steps is reached
        if log_step == max_steps:
            print(f"Max step {max_steps} reached! Stopping training...")
            break

In [ ]:
# INFERENCE
# Install minimal dependencies (`torch`, `transformers`, `timm`, `tokenizers`, ...)
# > pip install -r https://raw.githubusercontent.com/openvla/openvla/main/requirements-min.txt
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image

import torch

# Load Processor & VLA
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b", 
    attn_implementation="flash_attention_2",  # [Optional] Requires `flash_attn`
    torch_dtype=torch.bfloat16, 
    low_cpu_mem_usage=True, 
    trust_remote_code=True
).to("cuda:0")

# Grab image input & format prompt
image: Image.Image = get_from_camera(...)
prompt = "In: What action should the robot take to {<INSTRUCTION>}?\nOut:"

# Predict Action (7-DoF; un-normalize for BridgeData V2)
inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)
action = vla.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)

# Execute...
robot.act(action, ...)